Install OpenAI

In [0]:
%pip install openai

Load OpenAI Client

In [0]:
from openai import OpenAI

openai_key = dbutils.secrets.get(scope="my-project-secrets", key="openai-api-key")
client = OpenAI(api_key=openai_key)

Define the tools

In [0]:
def get_historical_trend(store_id: int, days: int = 30):
    """Get recent sales trend for a store."""
    df = spark.sql(f"""
        SELECT Date, Sales, Promo, IsStateHoliday
        FROM retail_project.gold.sales_features
        WHERE Store = {store_id}
        ORDER BY Date DESC
        LIMIT {days}
    """)
    pdf = df.toPandas()
    return {
        "store_id": store_id,
        "avg_sales": round(pdf["Sales"].mean(), 2),
        "min_sales": int(pdf["Sales"].min()),
        "max_sales": int(pdf["Sales"].max()),
        "days_analyzed": len(pdf)
    }

def get_forecast_summary(store_id: int):
    """Get model performance context for a store (using your Prophet/XGBoost results)."""
    # Simplified: returns your actual computed metrics as context
    # In a full production system, this would call the registered model directly
    return {
        "store_id": store_id,
        "model": "XGBoost (all-store) or Prophet (per-store)",
        "typical_error_pct": "9-10% MAPE based on holdout evaluation",
        "note": "Forecast accuracy varies by store; promotions and weather are significant factors."
    }

In [0]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_historical_trend",
            "description": "Get recent sales trend statistics for a specific store",
            "parameters": {
                "type": "object",
                "properties": {
                    "store_id": {"type": "integer", "description": "The store ID number"},
                    "days": {"type": "integer", "description": "Number of recent days to analyze, default 30"}
                },
                "required": ["store_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_forecast_summary",
            "description": "Get forecasting model accuracy context for a specific store",
            "parameters": {
                "type": "object",
                "properties": {
                    "store_id": {"type": "integer", "description": "The store ID number"}
                },
                "required": ["store_id"]
            }
        }
    }
]

Agent function

In [0]:
import json

def ask_agent(user_question: str):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a retail analytics assistant. You have access to tools "
                "that query real sales data and forecasting model results. "
                "Use the tools to answer questions accurately, and explain the "
                "numbers in plain language for a business stakeholder."
            )
        },
        {"role": "user", "content": user_question}
    ]

    # First call - let the model decide if/which tool to use
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools
    )

    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    if tool_calls:
        messages.append(response_message)

        available_functions = {
            "get_historical_trend": get_historical_trend,
            "get_forecast_summary": get_forecast_summary
        }

        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            function_to_call = available_functions[function_name]
            function_response = function_to_call(**function_args)

            messages.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": json.dumps(function_response)
            })

        # Second call - let the model generate a natural language answer using the tool results
        second_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )
        return second_response.choices[0].message.content
    else:
        return response_message.content

Test the agent with a real question

In [0]:
answer = ask_agent("What's the recent sales trend for store 1, and how accurate is our forecasting model for it?")
print(answer)

Test 2

In [0]:
print(ask_agent("Compare the sales performance of store 5 versus store 10 over the last 60 days"))

Test 3

In [0]:
print(ask_agent("What's the sales trend for store 1, and how accurate is our forecasting model for it"))

## GenAI Agent: Natural Language Analytics Assistant

Uses OpenAI function-calling to let users ask plain-language questions about
store performance and forecasting accuracy. The agent decides which data-query
tools to call, retrieves real results from the Gold table, and explains them
in business-friendly language.

### Example 1: Single-store trend + forecast accuracy
**Q:** "What's the recent sales trend for store 1, and how accurate is our
forecasting model for it?"
*(answer as shown above)*

### Example 2: Multi-store comparison (agent calls the same tool twice)
**Q:** "Compare the sales performance of store 5 versus store 10 over the
last 60 days"
*(answer as shown above)*

### Example 3: Rephrased question, consistent grounded answer
**Q:** "What's the sales trend for store 1, and how accurate is our
forecasting model for it"
*(answer as shown above)*

### Key capability demonstrated
The agent autonomously determines which tool(s) to call and how many times,
based on the natural language question, and produces consistent answers
grounded in real data rather than hallucinated figures.